# Análisis de experimentos WebGazer con pyxations

Este notebook reemplaza el pipeline custom del repo por las herramientas de **pyxations**.

Cubre ambos experimentos:
- `precision_experiment/`
- `antisaccade_experiment/`

Lo que pyxations hace por nosotros:
- Parsea los CSV de WebGazer (JSON embebido → DataFrame de muestras)
- Corre REMoDNaV (detección fijaciones/sacadas) como API Python, sin `os.system()`
- Guarda todo en formato BIDS + feather (un archivo por sujeto, no 1000+ TSVs)
- Segmentación de trials con `PreProcessing`

Lo que sigue siendo código custom:
- Lógica cognitiva antisacada (clasificación errores, reaction times)
- Normalización de coordenadas relativa a baseline
- Métricas de precisión específicas del experimento

## 0. Setup

In [2]:
# IMPORTANTE: usar python3.10 (no el python3 del sistema que es 3.8)
# En terminal: python3.10 -m jupyter notebook

# Instalar pyxations desde la versión local (descomentar la primera vez):
!pip install -e /home/gus/Documents/REPOS/pyxations

from pathlib import Path
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.feather as feather

from pyxations.bids_formatting import dataset_to_bids, compute_derivatives_for_dataset

print('pyxations OK')

REPO_ROOT = Path('.').resolve()
print(f'Working dir: {REPO_ROOT}')


Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///home/gus/Documents/REPOS/pyxations
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyxations (pyproject.toml) ... done
  Created wheel for pyxations: filename=pyxations-0.2.9-py3-none-any.whl size=4095 sha256=e2441f6f4cca63e23955d8a47e24fded66ad244657e5c327f99ee52d1757513f
  Stored in directory: /tmp/pip-ephem-wheel-cache-va11tkyy/wheels/17/0e/fc/69026a9c7df2d51824e72c9663e35863f9e2f79ae3d115a91f
Successfully built pyxations
  Attempting uninstall: pyxations
    Found existing installation: pyxations 0.2.9
    Uninstalling pyxations-0.2.9:
      Successfully uninstalled pyxations-0.2.9

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, ru

---
## 1. Experimento antisacada

### 1.1 Preparar archivos para pyxations

pyxations espera que los archivos se llamen `{subject_id}_{...}.csv`  
(sujeto primero, luego descripción).  
Los archivos originales son `antisacadas_{subject_id}.csv`, hay que reordenarlos.  
Hacemos copias en un directorio temporal — **no tocamos los originales**.

In [3]:
RAW_ANTI = REPO_ROOT / "antisaccade_experiment" / "raw_data"
ANTI_STAGING = REPO_ROOT / "antisaccade_experiment" / "_staging"  # temporal
ANTI_STAGING.mkdir(exist_ok=True)

for csv_file in RAW_ANTI.glob("antisacadas_*.csv"):
    subject_id = csv_file.stem.split("_")[1]          # "100", "101", ...
    new_name = f"{subject_id}_antisacadas.csv"         # "100_antisacadas.csv"
    dest = ANTI_STAGING / new_name
    if not dest.exists():
        shutil.copy(csv_file, dest)

print(f"Archivos preparados: {len(list(ANTI_STAGING.glob('*.csv')))} sujetos")
print([f.name for f in sorted(ANTI_STAGING.glob('*.csv'))][:5], "...")

Archivos preparados: 26 sujetos
['100_antisacadas.csv', '101_antisacadas.csv', '102_antisacadas.csv', '103_antisacadas.csv', '104_antisacadas.csv'] ...


### 1.2 Convertir a BIDS

In [4]:
ANTI_BIDS = REPO_ROOT / "antisaccade_experiment" / "bids"

bids_path = dataset_to_bids(
    target_folder_path=ANTI_BIDS,
    files_folder_path=ANTI_STAGING,
    dataset_name="antisaccade",
    session_substrings=1,
    format_name="webgazer",
)

print(f"BIDS creado en: {bids_path}")

# Verificar estructura
participants = pd.read_csv(bids_path / "participants.tsv", sep="\t")
print(f"\nParticipantes ({len(participants)}):")
print(participants.head())

BIDS creado en: /home/gus/Documents/REPOS/et_webcam_experiments/antisaccade_experiment/bids/antisaccade

Participantes (26):
   subject_id  old_subject_id
0           1              91
1           2              92
2           3              98
3           4              99
4           5             100


### 1.3 Calcular derivatives (detección de fijaciones y sacadas)

Reemplaza el loop de `os.system("remodnav ...")` que generaba ~1000 TSVs.  
pyxations corre REMoDNaV en paralelo sobre todos los sujetos y guarda en feather.

In [5]:
# behavioral_columns propaga metadata del experimento a samples.feather.
# pyxations lee el CSV de behavioral/ (ya copiado en el paso BIDS) y hace
# el join por trial_index — no hace falta leer el CSV manualmente después.
compute_derivatives_for_dataset(
    bids_dataset_folder=bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    num_processes=4,
    behavioral_columns=['typeOfSaccade', 'cueShownAtLeft', 'intraEnd',
                        'isTutorial', 'isSaccadeExperiment'],
    overwrite=True,   # re-correr para incluir las nuevas columnas
)

derivatives_path = Path(str(bids_path) + '_derivatives')
print(f'Derivatives en: {derivatives_path}')



Running eye movements detection for None eye...

Running eye movements detection for None eye...

Running eye movements detection for None eye...

Running eye movements detection for None eye...
Dropping saccades with average vel > 1000.0 deg/s, and fixations with amplitude > 1.5 deg
Kept 74 out of 231 fixations
Kept 234 out of 234 saccades
Finding previous and next saccades


100%|█████████████████████████████████████████| 74/74 [00:00<00:00, 1480.11it/s]



Kept 69 fixations with previous saccade
Computing average pupil size, and x and y position


  0%|                                                    | 0/69 [00:00<?, ?it/s]

Dropping saccades with average vel > 1000.0 deg/s, and fixations with amplitude > 1.5 deg
Kept 118 out of 396 fixations
Kept 397 out of 397 saccades
Finding previous and next saccades
Dropping saccades with average vel > 1000.0 deg/s, and fixations with amplitude > 1.5 deg

...

Skipping preprocessing: not enough parameters for trial segmentation (need (start_times & end_times) or (start_msgs & durations) or (start_msgs & end_msgs)).
Dropping saccades with average vel > 1000.0 deg/s, and fixations with amplitude > 1.5 deg
Kept 310 out of 789 fixations
Kept 796 out of 796 saccades
Finding previous and next saccades


100%|███████████████████████████████████████| 310/310 [00:00<00:00, 1891.82it/s]



Kept 301 fixations with previous saccade
Computing average pupil size, and x and y position


  0%|                                                   | 0/301 [00:00<?, ?it/s]/home/gus/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/gus/.local/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/gus/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/gus/.local/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/gus/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/gus/.local/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeW

Skipping preprocessing: not enough parameters for trial segmentation (need (start_times & end_times) or (start_msgs & durations) or (start_msgs & end_msgs)).
Derivatives en: /home/gus/Documents/REPOS/et_webcam_experiments/antisaccade_experiment/bids/antisaccade_derivatives


### 1.4 Cargar muestras de un sujeto

pyxations guarda las muestras parseadas en `samples.feather` (columnas: `trial_index`, `X`, `Y`, `tSample`).  
Usamos eso en lugar de parsear el JSON a mano.

In [6]:
def load_subject_samples(derivatives_path, subject_id):
    """
    Carga las muestras de gaze de un sujeto desde los derivatives de pyxations.

    Estructura:
        sub-XXXX/ses-YYY/samples.feather
            columnas: trial_index, X, Y, tSample, t_acum, ...
    """
    sub_path = derivatives_path / f"sub-{subject_id}"
    sessions = [s for s in sub_path.iterdir() if s.name.startswith("ses-")]
    if not sessions:
        raise FileNotFoundError(f"No sessions found for sub-{subject_id}")
    ses_path = sessions[0]
    return feather.read_feather(ses_path / "samples.feather").reset_index(drop=True)


# Ejemplo: sujeto 0001
samples = load_subject_samples(derivatives_path, "0001")
print(f"Muestras: {len(samples)} filas")
print(f"Columnas: {list(samples.columns)}")
print(f"trial_index únicos: {samples['trial_index'].nunique()}")
print(samples.head(3))

Muestras: 25679 filas
Columnas: ['line_number', 'trial_index', 'time_elapsed', 'X', 'Y', 'tSample', 't_acum', 'typeOfSaccade', 'cueShownAtLeft', 'intraEnd', 'isTutorial', 'isSaccadeExperiment']
trial_index únicos: 400
   line_number  trial_index  time_elapsed     X    Y  tSample  t_acum  \
0           29           29        156458  1059  510        0  156458   
1           29           29        156458  1078  480       73  156531   
2           29           29        156458  1119  471      142  156600   

  typeOfSaccade cueShownAtLeft  intraEnd isTutorial isSaccadeExperiment  
0          None           None       NaN       None                None  
1          None           None       NaN       None                None  
2          None           None       NaN       None                None  


### 1.5 Metadata experimental integrada en samples

Gracias a `behavioral_columns`, `samples.feather` ya contiene las columnas
del experimento (`typeOfSaccade`, `cueShownAtLeft`, `intraEnd`, etc.).
No hace falta ningún `pd.read_csv` adicional para el análisis.


In [7]:
# Mapeo subject_id (BIDS) → old_subject_id (nombre original)
participants = pd.read_csv(bids_path / 'participants.tsv', sep='\t', dtype=str)
id_map = dict(zip(participants['subject_id'], participants['old_subject_id']))

# Verificar que las columnas de behavioral estén en samples
sub_samples = load_subject_samples(derivatives_path, '0001')
print('Columnas en samples.feather:')
print([c for c in sub_samples.columns if c not in ['X','Y','tSample','t_acum','line_number','time_elapsed']])


Columnas en samples.feather:
['trial_index', 'typeOfSaccade', 'cueShownAtLeft', 'intraEnd', 'isTutorial', 'isSaccadeExperiment']


### 1.6 Análisis antisacada

La función recibe únicamente `df_samples` — los metadatos del trial
(`typeOfSaccade`, `cueShownAtLeft`, `intraEnd`) ya están en el feather.
La lógica de detección (normalización + threshold crossing) es idéntica al original.


In [8]:
def analyze_antisaccade_subject(
    df_samples,
    sacc_type,
    baseline_start=-200.0,
    baseline_end=100.0,
    FILTER=1.5,
    threshold=0.5,
    savgol_flag=False,
):
    """
    Análisis antisacada usando muestras de pyxations.

    Los metadatos del trial (typeOfSaccade, cueShownAtLeft, intraEnd) ya están
    en df_samples gracias a behavioral_columns en compute_derivatives_for_dataset.
    No se lee ningún CSV adicional.
    """
    from scipy.signal import savgol_filter

    # Metadatos únicos por trial desde las propias muestras
    meta_cols = ['trial_index', 'typeOfSaccade', 'cueShownAtLeft', 'intraEnd',
                 'isTutorial', 'isSaccadeExperiment']
    available = [c for c in meta_cols if c in df_samples.columns]
    trial_meta = df_samples[available].drop_duplicates('trial_index')

    mask = trial_meta['typeOfSaccade'] == sacc_type
    if 'isTutorial' in trial_meta.columns:
        mask &= trial_meta['isTutorial'] == False
    if 'isSaccadeExperiment' in trial_meta.columns:
        mask &= trial_meta['isSaccadeExperiment'].notna()
    sub_trials = trial_meta[mask]

    errors = 0
    rt_errors = []
    rt_correct = []
    n_rejected = 0
    ts_xs_ys = []

    for _, trial in sub_trials.iterrows():
        trial_idx = trial['trial_index']
        intra_end = trial['intraEnd']
        cue_left  = bool(trial['cueShownAtLeft'])

        t_samps = df_samples[df_samples['trial_index'] == trial_idx]
        if t_samps.empty:
            n_rejected += 1
            continue

        xs = t_samps['X'].to_numpy(dtype=float)
        ys = t_samps['Y'].to_numpy(dtype=float)
        ts = t_samps['tSample'].to_numpy(dtype=float) - intra_end

        mask_base = (ts > baseline_start) & (ts < baseline_end)
        mask_ref  = (ts > 500) & (ts <= 700)
        if not mask_base.any() or not mask_ref.any():
            n_rejected += 1
            continue

        x_base = np.median(xs[mask_base])
        x_ref  = np.median(xs[mask_ref])
        y_base = np.median(ys[mask_base])
        y_ref  = np.median(ys[mask_ref])

        denom_x = np.abs(x_base - x_ref)
        denom_y = np.abs(y_base - y_ref)
        if denom_x == 0 or denom_y == 0:
            n_rejected += 1
            continue

        xs = (xs - x_base) / denom_x
        ys = (ys - y_base) / denom_y

        if cue_left:
            xs = xs * -1

        if np.any(xs > FILTER) or np.any(xs < -FILTER):
            n_rejected += 1
            continue

        if savgol_flag:
            xs = savgol_filter(xs, 5, 2)

        xs_after = xs[ts > baseline_end]
        ts_after = ts[ts > baseline_end]

        if sacc_type == 'prosaccade':
            if np.any(xs_after < -threshold):
                errors += 1
                rt_errors.append(ts_after[np.where(xs_after < -threshold)[0][0]])
            else:
                idxs = np.where(xs_after >= threshold)[0]
                if len(idxs): rt_correct.append(ts_after[idxs[0]])
        else:
            if np.any(xs_after > threshold):
                errors += 1
                rt_errors.append(ts_after[np.where(xs_after > threshold)[0][0]])
            else:
                idxs = np.where(xs_after <= -threshold)[0]
                if len(idxs): rt_correct.append(ts_after[idxs[0]])

        ts_xs_ys.append((ts, xs, ys))

    n_valid = len(sub_trials) - n_rejected
    return {
        'n_total':           len(sub_trials),
        'n_rejected':        n_rejected,
        'n_valid':           n_valid,
        'error_rate':        errors / n_valid if n_valid else np.nan,
        'rt_errors':         rt_errors,
        'rt_error_median':   float(np.median(rt_errors)) if rt_errors else np.nan,
        'rt_correct':        rt_correct,
        'rt_correct_median': float(np.median(rt_correct)) if rt_correct else np.nan,
        'ts_xs_ys':          ts_xs_ys,
    }


# Ejemplo: sujeto 0001
sub_samples = load_subject_samples(derivatives_path, '0001')
for stype in ['prosaccade', 'antisaccade']:
    res = analyze_antisaccade_subject(sub_samples, stype)
    print(f'{stype}: n_valid={res["n_valid"]} | error_rate={res["error_rate"]:.3f} '
          f'| rt_correct={res["rt_correct_median"]:.0f}ms | rt_error={res["rt_error_median"]:.0f}ms')


prosaccade: n_valid=149 | error_rate=0.040 | rt_correct=407ms | rt_error=317ms
antisaccade: n_valid=136 | error_rate=0.059 | rt_correct=489ms | rt_error=386ms


### 1.7 Loop sobre todos los sujetos

In [9]:
all_results = []

for subject_id, old_id in id_map.items():
    try:
        sub_samples = load_subject_samples(derivatives_path, subject_id)
        # metadata ya en sub_samples — sin pd.read_csv adicional
        for stype in ['prosaccade', 'antisaccade']:
            res = analyze_antisaccade_subject(sub_samples, stype)
            all_results.append({
                'subject':           old_id,
                'type':              stype,
                'n_valid':           res['n_valid'],
                'n_rejected':        res['n_rejected'],
                'error_rate':        res['error_rate'],
                'rt_correct_median': res['rt_correct_median'],
                'rt_error_median':   res['rt_error_median'],
            })
    except Exception as e:
        print(f'[{old_id}] Error: {e}')

df_all = pd.DataFrame(all_results)
print(f'Sujetos procesados: {df_all["subject"].nunique()}')
print()
print(df_all.groupby('type')[['n_valid', 'error_rate', 'rt_correct_median']].mean().round(2))


Sujetos procesados: 26

             n_valid  error_rate  rt_correct_median
type                                               
antisaccade   134.92        0.06             454.69
prosaccade    136.35        0.05             418.81


### 1.9 Plot por sujeto (igual que el notebook original)

Señales normalizadas + KDE de reaction times.

In [10]:
import seaborn as sns

def plot_one_subject(ts_xs_ys, rt_errors, rt_correct, sacc_type, subject_id):
    """
    Replica el plot one_subject() del notebook original.
    ts_xs_ys: lista de (ts, xs, ys) — señales normalizadas por trial.
    rt_errors, rt_correct: listas de RTs en ms.
    """
    fig, axs = plt.subplots(3, 1, height_ratios=[1, 4, 1],
                            sharex=True, constrained_layout=True)

    for ts, xs, ys in ts_xs_ys:
        axs[1].plot(ts, xs, alpha=0.4, linewidth=0.8)

    axs[1].axhline(y=0.5,  color="k", linestyle="-")
    axs[1].axhline(y=-0.5, color="k", linestyle="-")
    axs[1].axvline(x=0,    color="k", linestyle="-")
    axs[1].set_ylabel("x coordinate (normalized)")
    axs[1].set_xticks(np.arange(-200, 1000, step=100))
    axs[1].set_xlim(-200, 700)
    axs[1].set_ylim(-2, 2)

    if sacc_type == "prosaccade":
        if rt_errors:
            sns.kdeplot(rt_errors, ax=axs[2])
            sns.rugplot(rt_errors, ax=axs[2], height=0.1)
        if rt_correct:
            sns.kdeplot(rt_correct, ax=axs[0])
            sns.rugplot(rt_correct, ax=axs[0], height=0.1)
        axs[2].invert_yaxis()
    else:
        if rt_errors:
            sns.kdeplot(rt_errors, ax=axs[0])
            sns.rugplot(rt_errors, ax=axs[0], height=0.1)
        if rt_correct:
            sns.kdeplot(rt_correct, ax=axs[2])
            sns.rugplot(rt_correct, ax=axs[2], height=0.1)
        axs[2].invert_yaxis()

    for ax in [axs[0], axs[2]]:
        ax.set_xlim(-200, 700)

    plt.suptitle(f"{sacc_type} — sujeto {subject_id}")
    fig.supxlabel("time (ms)")
    plt.show()


In [11]:
# Plot de un sujeto — cambiar SUBJECT_ID según el que quieras ver
SUBJECT_ID = "0001"
old_id = id_map[SUBJECT_ID]

sub_samples = load_subject_samples(derivatives_path, SUBJECT_ID)
sub_trials, _ = load_trial_metadata(RAW_ANTI, old_id)

for stype in ["prosaccade", "antisaccade"]:
    res = analyze_antisaccade_subject(sub_trials, sub_samples, stype)
    plot_one_subject(
        ts_xs_ys=res["ts_xs_ys"],
        rt_errors=res["rt_errors"],
        rt_correct=res["rt_correct"],
        sacc_type=stype,
        subject_id=old_id,
    )


NameError: name 'load_trial_metadata' is not defined

### 1.10 Boxplots grupales (error rate y RT)

In [ ]:
from scipy.stats import ranksums

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Error rate
ax = axes[0]
sns.boxplot(data=df_all, x="type", y="error_rate", width=0.3, ax=ax)
for patch in ax.patches: patch.set_facecolor("w")
p = ranksums(
    df_all[df_all["type"]=="prosaccade"]["error_rate"].dropna(),
    df_all[df_all["type"]=="antisaccade"]["error_rate"].dropna()
)[1]
ax.set_title(f"Error rate (p={p:.4f})")
ax.set_ylabel("% trials incorrectos")

# RT correcto
ax = axes[1]
sns.boxplot(data=df_all, x="type", y="rt_correct_median", width=0.3, ax=ax)
for patch in ax.patches: patch.set_facecolor("w")
p = ranksums(
    df_all[df_all["type"]=="prosaccade"]["rt_correct_median"].dropna(),
    df_all[df_all["type"]=="antisaccade"]["rt_correct_median"].dropna()
)[1]
ax.set_title(f"RT correcto mediana (p={p:.4f})")
ax.set_ylabel("time (ms)")

plt.tight_layout()
plt.show()

print(df_all.groupby("type")[["error_rate","rt_correct_median","rt_error_median"]].describe().round(1))


### 1.8 Resumen grupal

In [ ]:
df_all.groupby("type")[["n_valid", "error_rate", "rt_correct_median", "rt_error_median"]].mean().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
                                              
for ax, col, label in zip(axes,             
                         ["error_rate", "rt_correct_median"],               
                         ["Error rate", "RT correcto mediana (ms)"]):       
  for stype, grp in df_all.groupby("type"):                                 
      ax.hist(grp[col].dropna(), bins=8, alpha=0.6, label=stype)            
  ax.set_title(label)                                                       
  ax.legend()                                                               

plt.tight_layout()                                                            
plt.show()      

---
## 2. Experimento de precisión

### 2.1 Preparar archivos para pyxations

Los archivos ya siguen el formato `{subject}_{session}.csv`  
(ej: `gus_webcam4.csv`, `juan_webcam9_repe.csv`).


In [ ]:
RAW_PREC = REPO_ROOT / "precision_experiment" / "raw_data" / "precision-antisaccade-exp"
PREC_STAGING = REPO_ROOT / "precision_experiment" / "_staging"
PREC_STAGING.mkdir(exist_ok=True)

raw_files = sorted(RAW_PREC.glob("*.csv"))
print(f"Archivos encontrados: {len(raw_files)}")
print([f.name for f in raw_files])


In [ ]:
# Copiar al staging (los nombres ya siguen el formato subject_session.csv)
for csv_file in raw_files:
    dest = PREC_STAGING / csv_file.name
    if not dest.exists():
        shutil.copy(csv_file, dest)

print(f"Archivos en staging: {len(list(PREC_STAGING.glob('*.csv')))}")


### 2.2 Convertir a BIDS y calcular derivatives


In [ ]:
PREC_BIDS = REPO_ROOT / 'precision_experiment' / 'bids'

prec_bids_path = dataset_to_bids(
    target_folder_path=PREC_BIDS,
    files_folder_path=PREC_STAGING,
    dataset_name='precision',
    session_substrings=2,
    format_name='webgazer',
)

compute_derivatives_for_dataset(
    bids_dataset_folder=prec_bids_path,
    dataset_format='webgazer',
    detection_algorithm='remodnav',
    num_processes=4,
    behavioral_columns=['trial-tag', 'center_x', 'center_y', 'start-x', 'start-y'],
    overwrite=True,
)

prec_derivatives_path = Path(str(prec_bids_path) + '_derivatives')
print(f'Derivatives en: {prec_derivatives_path}')


In [ ]:
# Mapa subject_id (BIDS) -> old_id (nombre original)
participants = pd.read_csv(prec_bids_path / "participants.tsv", sep="\t", dtype=str)
prec_id_map = dict(zip(participants["subject_id"], participants["old_subject_id"]))
print("Subjects:", prec_id_map)


### 2.3 Análisis de precisión

Las columnas `trial-tag`, `center_x`, `center_y`, `start-x`, `start-y`
ya están en `samples.feather` gracias a `behavioral_columns`.
La función no lee ningún CSV — trabaja únicamente con el feather de pyxations.


In [ ]:
def analyze_precision_subject(samples_path,
                               trial_tag='validation-stimulus',
                               first_sample=500, max_var=75):
    """
    Métricas de precisión usando muestras de pyxations.
    Las columnas trial-tag, center_x, center_y, start-x, start-y
    ya están en samples.feather — no se lee ningún CSV adicional.
    """
    samples = feather.read_feather(samples_path)

    # Trials de validación directamente desde samples
    val_samples = samples[samples['trial-tag'] == trial_tag]
    val_trials = (
        val_samples[['trial_index', 'center_x', 'center_y', 'start-x', 'start-y']]
        .drop_duplicates('trial_index')
    )

    results = []
    for _, row in val_trials.iterrows():
        tidx        = row['trial_index']
        presented_x = row['center_x'] + row['start-x']
        presented_y = row['center_y'] + row['start-y']

        trial_samps = samples[
            (samples['trial_index'] == tidx) & (samples['tSample'] > first_sample)
        ]
        if trial_samps.empty:
            results.append(dict(trial_index=tidx, presented_point_x=presented_x,
                                presented_point_y=presented_y, n_samples=0,
                                horizontal_error_px=np.nan, vertical_error_px=np.nan,
                                total_error_px=np.nan))
            continue

        xs = trial_samps['X'].values
        ys = trial_samps['Y'].values

        if np.std(xs) > max_var or np.std(ys) > max_var:
            results.append(dict(trial_index=tidx, presented_point_x=presented_x,
                                presented_point_y=presented_y, n_samples=len(trial_samps),
                                horizontal_error_px=np.nan, vertical_error_px=np.nan,
                                total_error_px=np.nan))
            continue

        results.append(dict(
            trial_index=tidx,
            presented_point_x=presented_x,
            presented_point_y=presented_y,
            n_samples=len(trial_samps),
            horizontal_error_px=np.abs(xs - presented_x).mean(),
            vertical_error_px=np.abs(ys - presented_y).mean(),
            total_error_px=np.sqrt((xs - presented_x)**2 + (ys - presented_y)**2).mean(),
        ))

    return pd.DataFrame(results)


In [ ]:
all_prec_results = {}

for subject_id, old_id in prec_id_map.items():
    sub_path = prec_derivatives_path / f'sub-{subject_id}'
    if not sub_path.exists():
        print(f'sub-{subject_id} no encontrado, saltando.')
        continue

    for ses_dir in sorted(sub_path.iterdir()):
        if not ses_dir.name.startswith('ses-'):
            continue
        ses_name = ses_dir.name[4:]
        samples_path = ses_dir / 'samples.feather'
        if not samples_path.exists():
            print(f'  No samples para sub-{subject_id} ses-{ses_name}, saltando.')
            continue

        print(f'Procesando sub-{subject_id} ({old_id}), ses-{ses_name}...')
        # sin pd.read_csv — toda la metadata viene del feather
        df_res = analyze_precision_subject(samples_path)
        df_res['subject_id'] = subject_id
        df_res['old_id'] = old_id
        df_res['session'] = ses_name

        key = f'{old_id}_{ses_name}'
        all_prec_results[key] = df_res

        n_valid = df_res['horizontal_error_px'].notna().sum()
        print(f'  Trials válidos: {n_valid}/{len(df_res)}')
        print(f'  Error horizontal: {df_res["horizontal_error_px"].mean():.1f} px')
        print(f'  Error vertical:   {df_res["vertical_error_px"].mean():.1f} px')
        print(f'  Error total:      {df_res["total_error_px"].mean():.1f} px\n')


### 2.4 Heatmap de error por punto de validación

Replica el heatmap del `precision_analysis.ipynb` original.  
Cada celda del heatmap es el error medio para ese punto de la grilla 3×3.


In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms


def confidence_ellipse(x, y, ax, n_std=3.0, facecolor="none", **kwargs):
    """Elipse de confianza para datos 2D."""
    if x.size != y.size:
        raise ValueError("x e y deben tener el mismo tamaño")
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2,
                      facecolor=facecolor, **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    mean_x = np.mean(x)
    scale_y = np.sqrt(cov[1, 1]) * n_std
    mean_y = np.mean(y)
    transf = transforms.Affine2D().rotate_deg(45).scale(scale_x, scale_y).translate(mean_x, mean_y)
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)


def plot_precision_heatmaps(all_prec_results, vmin=0, vmax=300, cmap="Reds"):
    """
    Heatmap 3×3 de error horizontal y vertical por punto de validación,
    para cada sujeto/sesión. Replica precision_analysis.ipynb celda 18.
    """
    keys = list(all_prec_results.keys())
    n = len(keys)
    fig, axes = plt.subplots(n, 2, figsize=(5, 2.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row_idx, key in enumerate(keys):
        df_res = all_prec_results[key]
        unique_xs = sorted(df_res["presented_point_x"].unique())
        unique_ys = sorted(df_res["presented_point_y"].unique(), reverse=True)

        for col_idx, (error_col, label) in enumerate([
            ("horizontal_error_px", "Horizontal"),
            ("vertical_error_px", "Vertical"),
        ]):
            grid = np.full((len(unique_ys), len(unique_xs)), np.nan)
            for i, py in enumerate(unique_ys):
                for j, px in enumerate(unique_xs):
                    mask = (df_res["presented_point_x"] == px) & (df_res["presented_point_y"] == py)
                    val = df_res.loc[mask, error_col].mean()
                    grid[i, j] = val

            ax = axes[row_idx, col_idx]
            im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
            ax.set_title(f"{key}\n{label} (px)", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            plt.colorbar(im, ax=ax, shrink=0.8)

    plt.suptitle("Error de precisión por punto de validación", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_precision_heatmaps(all_prec_results)


### 2.5 Error a lo largo del experimento

Scatter de error por trial con regresión, separado por bloque.  
Replica el `lmplot` de `precision_analysis.ipynb` original.


In [ ]:
from scipy.stats import linregress


def plot_precision_over_time(all_prec_results):
    """
    Error de precisión a lo largo del experimento por sujeto/sesión.
    Replica precision_analysis.ipynb celda 19.
    """
    palette = sns.color_palette("flare", 8)

    for key, df_res in all_prec_results.items():
        df_plot = df_res.dropna(subset=["horizontal_error_px"]).copy().reset_index(drop=True)
        df_plot["abs"] = df_plot.index + 1
        df_plot["block"] = ((df_plot["trial_index"] - 1) % 8 + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        for ax, (error_col, ylabel) in zip(axes, [
            ("horizontal_error_px", "Error horizontal (px)"),
            ("vertical_error_px", "Error vertical (px)"),
            ("total_error_px", "Error total (px)"),
        ]):
            for block_id in sorted(df_plot["block"].unique()):
                sub = df_plot[df_plot["block"] == block_id]
                color = palette[(block_id - 1) % len(palette)]
                ax.scatter(sub["abs"], sub[error_col], c=[color], alpha=0.4, s=8)

            slope, intercept, *_ = linregress(df_plot["abs"], df_plot[error_col])
            x_range = np.array([df_plot["abs"].min(), df_plot["abs"].max()])
            ax.plot(x_range, slope * x_range + intercept, "k-", linewidth=2)

            ax.vlines([i * 72 for i in range(1, 9)], 0, 600, colors="k",
                      linewidths=0.5, linestyles="dashed")
            ax.set_xlim(0, len(df_plot) + 5)
            ax.set_ylim(0, 600)
            ax.set_xlabel("Trial")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{key}")

        plt.suptitle(f"Error de precisión — {key}", fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.show()

        h = df_res['horizontal_error_px'].mean()
        v = df_res['vertical_error_px'].mean()
        t = df_res['total_error_px'].mean()
        print(f"{key}: H={h:.1f} px  V={v:.1f} px  Total={t:.1f} px")


plot_precision_over_time(all_prec_results)


### 2.6 Scatter de posición de gaze vs punto presentado

Para cada punto de validación, grafica todos los samples de gaze  
junto con elipses de confianza — igual al análisis en el notebook original.


In [ ]:
def plot_gaze_scatter(all_prec_results, screen_res=(1920, 1080)):
    """
    Scatter de posición de gaze para cada punto de validación.
    Muestra elipse de confianza (3 std) por punto.
    """
    for key, df_res in all_prec_results.items():
        csv_path = RAW_PREC / f"{df_res['old_id'].iloc[0]}_{df_res['session'].iloc[0]}.csv"
        df_raw = pd.read_csv(csv_path)
        val = df_raw[df_raw["trial-tag"] == "validation-stimulus"].copy()

        unique_pts = val[["start-x", "start-y"]].drop_duplicates()
        unique_pts["px"] = unique_pts["start-x"] + val["center_x"].iloc[0]
        unique_pts["py"] = unique_pts["start-y"] + val["center_y"].iloc[0]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.set_xlim(0, screen_res[0])
        ax.set_ylim(screen_res[1], 0)  # y invertido (0 arriba)
        ax.axvline(screen_res[0] / 2, color="gray", linewidth=0.5, linestyle="--")
        ax.axhline(screen_res[1] / 2, color="gray", linewidth=0.5, linestyle="--")

        # Samples de pyxations
        sub_path = prec_derivatives_path / f"sub-{df_res['subject_id'].iloc[0]}"
        ses_path = sub_path / f"ses-{df_res['session'].iloc[0]}"
        samples = feather.read_feather(ses_path / "samples.feather")

        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_pts)))
        for color, (_, pt) in zip(colors, unique_pts.iterrows()):
            # Trials para este punto
            pt_trials = val[
                (val["start-x"] == pt["start-x"]) & (val["start-y"] == pt["start-y"])
            ]["trial_index"].values
            pt_samples = samples[samples["trial_index"].isin(pt_trials)]
            xs = pt_samples["X"].values
            ys = pt_samples["Y"].values

            ax.scatter(xs, ys, s=3, alpha=0.2, color=color)
            ax.scatter(pt["px"], pt["py"], marker="+", s=200, linewidths=2, color=color,
                       label=f"({pt['px']:.0f}, {pt['py']:.0f})")

            if len(xs) > 2:
                confidence_ellipse(xs, ys, ax, n_std=2, edgecolor=color, linewidth=1.5)

        ax.set_title(f"{key} — gaze vs punto presentado", fontsize=10)
        ax.set_xlabel("X (px)")
        ax.set_ylabel("Y (px)")
        ax.legend(fontsize=7, loc="upper right")
        plt.tight_layout()
        plt.show()


plot_gaze_scatter(all_prec_results)


---
## Notas

- Los directorios `_staging/` son temporales. Se pueden borrar después de generar los BIDS.
- Los `_derivatives/` contienen todo el output de pyxations (feather por sujeto).
- Para re-correr la detección: pasar `overwrite=True` a `compute_derivatives_for_dataset`.
- La lógica de errores antisacada en la sección 1.6 es específica de la tarea —
  pyxations detecta los movimientos, pero la clasificación correcto/incorrecto
  sigue siendo responsabilidad del análisis.
- En sección 2, `tSample` del feather de pyxations = tiempo relativo al trial (ms),
  equivalente al campo `t` del JSON de WebGazer. El filtro `first_sample=500` es idéntico al original.
